In [ ]:
import pandas as pd

df = pd.read_csv("/Users/rajeev/Desktop/Coding/AQIproject/data/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69.csv")

df.head() #automatically shows only first 5 rows


In [ ]:
df.head(10)

In [ ]:
df.columns


In [ ]:
df.info()



In [ ]:
df.shape


In [ ]:
df['last_update'] = pd.to_datetime(df['last_update'])
df.head()
#This will convert pollutant rows into pollutant columns.
df_pivot = df.pivot_table(
    index=['country', 'state', 'city', 'station', 'last_update'],
    columns='pollutant_id',
    values='pollutant_avg'
).reset_index()

df_pivot.head(10)


In [ ]:
# Check missing values
df_pivot.isnull().sum()


In [ ]:
df_pivot.isnull().sum()
# Remove rows with any missing values
df_clean = df_pivot.dropna()

df_clean.shape



In [ ]:
# Define target variable
y = df_clean['PM2.5']

# Define features (drop non-numeric + target column)
X = df_clean[['CO', 'NH3', 'NO2', 'OZONE', 'PM10', 'SO2']]

X.head(10)


In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)
from sklearn.linear_model import LinearRegression

# Create model
lr = LinearRegression()

# Train model
lr.fit(X_train, y_train)

print("Model trained successfully")
y_pred = lr.predict(X_test)

y_pred[:5]
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create model
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Train model
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)


In [ ]:
# Function to calculate AQI from PM2.5 (CPCB standard)
def calculate_aqi_pm25(pm):
    
    if pm <= 30:
        BP_lo, BP_hi = 0, 30
        I_lo, I_hi = 0, 50
        
    elif pm <= 60:
        BP_lo, BP_hi = 31, 60
        I_lo, I_hi = 51, 100
        
    elif pm <= 90:
        BP_lo, BP_hi = 61, 90
        I_lo, I_hi = 101, 200
        
    elif pm <= 120:
        BP_lo, BP_hi = 91, 120
        I_lo, I_hi = 201, 300
        
    elif pm <= 250:
        BP_lo, BP_hi = 121, 250
        I_lo, I_hi = 301, 400
        
    else:
        BP_lo, BP_hi = 251, 500
        I_lo, I_hi = 401, 500
    
    aqi = ((I_hi - I_lo) / (BP_hi - BP_lo)) * (pm - BP_lo) + I_lo
    return round(aqi)

# Create AQI column
df_clean['AQI'] = df_clean['PM2.5'].apply(calculate_aqi_pm25)

df_clean[['PM2.5', 'AQI']].head()


In [ ]:
# Define new target
y = df_clean['AQI']

# Define features (exclude PM2.5 now because AQI is calculated from it)
X = df_clean[['CO', 'NH3', 'NO2', 'OZONE', 'PM10', 'SO2']]
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("AQI Prediction Results:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


In [ ]:
import pandas as pd

# Create comparison table
comparison = pd.DataFrame({
    "Actual AQI": y_test.values,
    "Predicted AQI": y_pred.round(2)
})

# Show first 10 predictions
comparison.head(10)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI")
plt.show()


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create model
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train model
gbr.fit(X_train, y_train)

# Predict
y_pred_gbr = gbr.predict(X_test)

# Evaluate
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Results:")
print("MAE:", mae_gbr)
print("RMSE:", rmse_gbr)
print("R2 Score:", r2_gbr)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importance
importance = gbr.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

# Plot
plt.figure(figsize=(6,4))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.gca().invert_yaxis()
plt.title("Feature Importance (Gradient Boosting)")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Remove weakest features
X_reduced = df_clean[['CO', 'NO2', 'OZONE', 'PM10']]

y = df_clean['AQI']

# Split again
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

# Train model
gbr_reduced = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr_reduced.fit(X_train_r, y_train_r)

# Predict
y_pred_r = gbr_reduced.predict(X_test_r)

# Evaluate
print("Reduced Feature Model Results:")
print("MAE:", mean_absolute_error(y_test_r, y_pred_r))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_r)))
print("R2 Score:", r2_score(y_test_r, y_pred_r))

In [ ]:
# Hybrid prediction (average of two models)
y_pred_hybrid = (y_pred + y_pred_gbr) / 2

# Evaluate hybrid model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Hybrid Model Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid)))
print("R2 Score:", r2_score(y_test, y_pred_hybrid))

In [ ]:
import matplotlib.pyplot as plt

# Model names
models = ['Linear Regression', 'Gradient Boosting', 'Hybrid Model']

# R2 scores (replace if needed with exact values)
r2_scores = [0.7259, 0.7326, 0.7599]

plt.figure(figsize=(6,4))
plt.bar(models, r2_scores)
plt.ylabel("R2 Score")
plt.title("Model Comparison (R2 Score)")
plt.ylim(0.6, 0.8)
plt.xticks(rotation=20)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

# Use same final features
X_final = df_clean[['CO', 'NO2', 'OZONE', 'PM10']]
y_final = df_clean['AQI']

# Create Linear Regression model
lr_model = LinearRegression()

# Perform 5-fold cross validation
cv_scores_lr = cross_val_score(
    lr_model,
    X_final,
    y_final,
    cv=5,
    scoring='r2'
)

print("Linear Regression Cross Validation R2 Scores:", cv_scores_lr)
print("Average R2:", np.mean(cv_scores_lr))

In [ ]:
def categorize_aqi(aqi):
    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Satisfactory"
    elif aqi <= 200:
        return "Moderate"
    elif aqi <= 300:
        return "Poor"
    elif aqi <= 400:
        return "Very Poor"
    else:
        return "Severe"

df_clean["AQI_Category"] = df_clean["AQI"].apply(categorize_aqi)

df_clean[["AQI", "AQI_Category"]].head()

In [ ]:
from sklearn.model_selection import train_test_split

X_class = df_clean[['CO', 'NO2', 'OZONE', 'PM10']]
y_class = df_clean['AQI_Category']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42
)

print("Classification data ready")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

clf = RandomForestClassifier(n_estimators=200, random_state=42)

clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("\nClassification Report:\n")
print(classification_report(y_test_c, y_pred_c))

In [ ]:
clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print(classification_report(y_test_c, y_pred_c))

In [ ]:
from sklearn.model_selection import train_test_split

# Features
X = df_clean[['CO', 'NH3', 'NO2', 'OZONE', 'PM10', 'SO2']]

# Target
y = df_clean['AQI']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train Test Split Ready")

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create model
ada = AdaBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

# Train model
ada.fit(X_train, y_train)

# Predict
y_pred_ada = ada.predict(X_test)

# Evaluate
print("AdaBoost Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_ada))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ada)))
print("R2 Score:", r2_score(y_test, y_pred_ada))

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create model
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Train model
gbr.fit(X_train, y_train)

# Predict
y_pred_gbr = gbr.predict(X_test)

# Evaluate
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Results:")
print("MAE:", mae_gbr)
print("RMSE:", rmse_gbr)
print("R2 Score:", r2_gbr)